In [1]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 2
with open("/home/chenzihao/workspace/cc2cc_test5/cc2cc/utils/gmtkn-def2.json") as f:
    json_data = json.load(f)

basis_args = "def2-TZVPD"
print(basis_args)

dft_type_list = ["scf_ene", "scf_d3bj_ene"]
data = pd.read_csv(
    "/home/chenzihao/workspace/cc2cc_test5/validate_hkqai/ccdft_def2-TZVPD_atom-1-1679342_gmtkn-def2.csv"
)

with open(f"./subset.json") as f:
    full_subset_dict = json.load(f)["full_subset_dict"]

for name_set, subset_list_ in full_subset_dict.items():
    full_subset_dict[name_set] = np.sort(subset_list_)
data_subset = {}

for name_set, subset_list_ in full_subset_dict.items():
    for dft_type in dft_type_list:
        if dft_type in data.columns:
            data_name = (data["name"].str.split(f"_{basis_args}").str[0]).to_numpy()
            data_dft = data[dft_type].to_numpy() * 627.5094733748099
        else:
            data_name = []
            for i_subset in subset_list_:
                if i_subset == "BH76RC":
                    data_name.append(json_data["molecule_BH76"])
                else:
                    data_name.append(json_data[f"molecule_{i_subset}"])
            data_name = np.concatenate(data_name)
            data_dft = np.zeros_like(data_name, dtype=float)

        for i_subset in subset_list_:
            name_subset = f"{dft_type}_{i_subset}"
            data_subset[name_subset] = {
                "name": [],
                "dft_name": [],
                "dft": [],
                "cc": [],
            }

            if i_subset == "BH76RC":
                molecular_list = json_data["molecule_BH76"]
            else:
                molecular_list = json_data[f"molecule_{i_subset}"]

            for i_molecule_name in molecular_list:
                col = np.where(data_name == i_molecule_name)[0]
                if col.size != 1:
                    # if verbose > 0:
                    #     print(f"Warning: {i_molecule_name} not found in data file")
                    continue

            reaction_dict = json_data[f"reaction-{i_subset}"]
            for i_reaction_name, i_reaction in reaction_dict.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                finished, exist = True, True

                for i in range(len(systems_list)):
                    mole_name = (
                        systems_list[i]
                        if i_subset == "BH76RC"
                        else f"{i_subset}-{systems_list[i]}"
                    )
                    stoichiometry = int(stoichiometry_list[i])

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished, exist = False, False
                        # if verbose > 0:
                        #     print(f"Warning: {mole_name} not found in json, ERROR")
                        break

                    col = np.where(data_name == mole_name)[0]
                    if col.size == 1:
                        atomic_energy_dft += data_dft[col[0]] * stoichiometry
                    else:
                        finished = False
                        # if verbose > 0:
                        #     print(f"Warning: {mole_name} not found in data csv file")
                        break

                if exist:
                    data_subset[name_subset]["name"].append(i_reaction_name)
                if finished:
                    data_subset[name_subset]["dft_name"].append(i_reaction_name)
                    for i in range(len(systems_list)):
                        data_subset[name_subset]["dft_name"][
                            -1
                        ] = f"{data_subset[name_subset]["dft_name"][-1]}-{systems_list[i]}"

                    atomic_energy_cc = i_reaction["reference"]
                    data_subset[name_subset]["dft"].append(
                        abs(atomic_energy_dft - atomic_energy_cc)
                    )
                    data_subset[name_subset]["cc"].append(abs(atomic_energy_cc))
                    if np.abs(atomic_energy_cc) > 1000:
                        print(
                            f"Warning: {i_reaction_name} in {name_subset} has a large CC energy: {atomic_energy_cc} kcal/mol"
                        )

            for key, val in data_subset.items():
                for key2, val2 in val.items():
                    if isinstance(val2, list):
                        data_subset[key][key2] = np.array(val2)

            if verbose > 1:
                argsort_atomic_energy_dft = np.argsort(data_subset[name_subset]["dft"])[
                    ::-1
                ]
                for i in range(len(argsort_atomic_energy_dft)):
                    print(
                        f"Top {i+1} with name {data_subset[name_subset]["dft_name"][argsort_atomic_energy_dft[i]]} DFT: {data_subset[name_subset]['dft'][argsort_atomic_energy_dft[i]]} kcal/mol"
                    )

header = dft_type_list + ["Processed"]

df_summary_subset = pd.DataFrame(columns=header)
mean_subset = pd.DataFrame(columns=header)
wtmad_1_subset = pd.DataFrame(columns=header)
wtmad_2_subset = pd.DataFrame(columns=header)

for dft_type in dft_type_list:
    mean_absolute_deviation_list = []

    for name_set, subset_list_ in full_subset_dict.items():
        subset_dft = []
        wtmad_1_dft = []
        wtmad_2_dft = []
        processed = []

        for i_subset in subset_list_:
            name_subset = f"{dft_type}_{i_subset}"

            if len(data_subset[name_subset]["dft"]) == 0:
                df_summary_subset.loc[i_subset, dft_type] = 0
                df_summary_subset.loc[i_subset, "Processed"] = (
                    f"0 / {len(data_subset[name_subset]['name'])}"
                )
            else:
                df_summary_subset.loc[i_subset, dft_type] = np.mean(
                    data_subset[name_subset]["dft"]
                )
                df_summary_subset.loc[i_subset, "Processed"] = (
                    "DONE"
                    if (
                        (
                            len(data_subset[name_subset]["dft"])
                            == len(data_subset[name_subset]["name"])
                        )
                        and (len(data_subset[name_subset]["dft"]) != 0)
                    )
                    else f"{len(data_subset[name_subset]['dft'])} / "
                    f"{len(data_subset[name_subset]['name'])}"
                )

                if np.mean(data_subset[name_subset]["cc"]) > 75:
                    wtmad_1 = 0.1
                elif np.mean(data_subset[name_subset]["cc"]) < 7.5:
                    wtmad_1 = 10
                else:
                    wtmad_1 = 1

                subset_dft = np.append(subset_dft, data_subset[name_subset]["dft"])
                wtmad_1_dft = np.append(
                    wtmad_1_dft,
                    wtmad_1 * np.mean(data_subset[name_subset]["dft"]),
                )
                wtmad_2_dft = np.append(
                    wtmad_2_dft,
                    data_subset[name_subset]["dft"]
                    / np.mean(data_subset[name_subset]["cc"]),
                )
                mean_absolute_deviation_list = np.append(
                    mean_absolute_deviation_list,
                    data_subset[name_subset]["cc"],
                )
            if (
                len(data_subset[name_subset]["dft"])
                == len(data_subset[name_subset]["name"])
                and len(data_subset[name_subset]["name"]) > 0
            ):
                processed.append(1)
            else:
                processed.append(0)

        mean_subset.loc[name_set, dft_type] = np.mean(subset_dft)
        wtmad_1_subset.loc[name_set, dft_type] = np.mean(wtmad_1_dft)
        wtmad_2_subset.loc[name_set, dft_type] = np.sum(wtmad_2_dft)
        mean_subset.loc[name_set, "Processed"] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_1_subset.loc[name_set, "Processed"] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_2_subset.loc[name_set, "Processed"] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )

    mean_absolute_deviation = 56.84 / len(mean_absolute_deviation_list)
    print(
        f"Mean absolute deviation for {dft_type}: {mean_absolute_deviation:.4f} kcal/mol"
    )
    for name_set in full_subset_dict.keys():
        wtmad_2_subset.loc[name_set, dft_type] = (
            mean_absolute_deviation * wtmad_2_subset.loc[name_set, dft_type]
        )

    wtmad_1_subset.loc["summary", "Processed"] = "--"
    wtmad_2_subset.loc["summary", "Processed"] = "--"
    wtmad_1_subset.loc["summary", dft_type] = 0
    wtmad_2_subset.loc["summary", dft_type] = 0
    for name_set in full_subset_dict.keys():
        wtmad_1_subset.loc["summary", dft_type] += wtmad_1_subset.loc[
            name_set, dft_type
        ]
        wtmad_2_subset.loc["summary", dft_type] += wtmad_2_subset.loc[
            name_set, dft_type
        ]

print("MAE")
display(mean_subset)
print("wtmad_1")
display(wtmad_1_subset)
print("wtmad_2")
display(wtmad_2_subset)
print("Summary of Subset")
print("MAE")
display(df_summary_subset)

# save summary to excel with date
df_summary_subset.to_excel(f"../validate_hkqai_done/summary_subset_{date}.xlsx")
mean_subset.to_excel(f"../validate_hkqai_done/mean_subset_{date}.xlsx")
wtmad_1_subset.to_excel(f"../validate_hkqai_done/wtmad_1_subset_{date}.xlsx")
wtmad_2_subset.to_excel(f"../validate_hkqai_done/wtmad_2_subset_{date}.xlsx")

def2-TZVPD
Top 1 with name 5-al2me6-alme3 DFT: 5.31837773686275 kcal/mol
Top 2 with name 4-al2me5-alme2-alme3 DFT: 3.70506898749154 kcal/mol
Top 3 with name 1-al2f6-alf3 DFT: 3.5393704901449397 kcal/mol
Top 4 with name 3-al2me4-alme2 DFT: 0.9957893619779483 kcal/mol
Top 5 with name 2-al2cl6-alcl3 DFT: 0.7178096454590559 kcal/mol
Top 6 with name 0-al2h6-alh3 DFT: 0.3802605289965868 kcal/mol
Top 1 with name 8-mgs-mg-s DFT: 9.271880338742633 kcal/mol
Top 2 with name 7-mgo-mg-o DFT: 8.49468604504655 kcal/mol
Top 3 with name 0-bef-be-f DFT: 7.81062633810069 kcal/mol
Top 4 with name 1-beo-be-o DFT: 7.2427535862385355 kcal/mol
Top 5 with name 3-hf-h-f DFT: 5.592363692283101 kcal/mol
Top 6 with name 9-nao-na-o DFT: 4.113320584964825 kcal/mol
Top 7 with name 4-kf-k-f DFT: 3.3684083761691 kcal/mol
Top 8 with name 5-lif-li-f DFT: 2.8043342337070385 kcal/mol
Top 9 with name 2-cao-ca-o DFT: 1.6419373133510788 kcal/mol
Top 10 with name 6-lio-li-o DFT: 0.7460932462854544 kcal/mol
Top 1 with name 2-c2

/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packa

,scf_ene,scf_d3bj_ene,Processed
sub1,9.865944,8.813946,9 / 18
sub2,13.907246,5.341473,1 / 9
sub3,NaN,NaN,0 / 7
sub4,2.047967,0.741035,0 / 12
sub5,1.021742,0.553854,0 / 9


wtmad_1


,scf_ene,scf_d3bj_ene,Processed
sub1,4.294093,3.69959,9 / 18
sub2,6.425508,2.646844,1 / 9
sub3,NaN,NaN,0 / 7
sub4,20.479667,7.410351,0 / 12
sub5,11.63909,5.762892,0 / 9
summary,NaN,NaN,--


wtmad_2


,scf_ene,scf_d3bj_ene,Processed
sub1,3.335045,3.016844,9 / 18
sub2,3.069999,0.882021,1 / 9
sub3,0.0,0.0,0 / 7
sub4,2.136446,0.77305,0 / 12
sub5,1.561966,0.903288,0 / 9
summary,10.103456,5.575203,--


Summary of Subset
MAE


,scf_ene,scf_d3bj_ene,Processed
AL2X6,2.442779,4.355891,DONE
ALK8,0,0,0 / 8
ALKBDE10,5.10864,4.646737,DONE
BH76RC,0,0,0 / 30
DC13,0,0,0 / 13
DIPCS10,6.418703,6.403969,DONE
FH51,0,0,0 / 51
G21EA,5.291996,5.287058,DONE
G21IP,3.825676,3.83182,DONE
G2RC,0,0,0 / 25
